In [3]:
import requests
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import statsmodels.api as sm
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
import scipy.stats as stats
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import GridSearchCV
import json

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


## Part1 -- Backup Data Loading

In [2]:
# load backup data
def process_backup_data(file_path):
    column_names = ['Date+T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
    col_after = ['Date', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
    col1 = ['T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
    col2 = ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']

    data = pd.read_csv(file_path, header=None, names=column_names)

    # check if 'T1' is concatenated with 'Date' and handle accordingly
    if '-' in data.iloc[0, 0]:
        # split 'Date+T1' into two separate columns 'Date' and 'T1'
        new_cols = data['Date+T1'].str.split(' T1:', expand=True)
        data['Date'] = new_cols[0]
        data['T1'] = new_cols[1].astype(float)
        data = data.drop(columns='Date+T1')
        # make sure 'T1' is correctly placed without affecting other columns
        data = data[col_after]
        for col in data[col1]:
            data[col] = data[col].str.split(':').str[1].astype(float)
    else:
        # separate 'Date' and 'T1' if not concatenated
        data= data.reset_index().rename(columns={'index': 'Date'})
        data = data.rename(columns={'Date+T1': 'T1'})
        data = data[col_after]
        for col in data[col2]:
            data[col] = data[col].str.split(':').str[1].astype(float)


    data.drop(data.index[:1], inplace=True)
    return data



In [3]:

# concat all backup data 
backup0 = 'data_backup0.csv'
backup1 = 'data_backup1.csv'
backup2 = 'data_backup2.csv'
backup3 = 'data_backup3.csv'
backup4 = 'data_backup4.csv'
twentyfour_apr_may = process_backup_data(backup0)
twentyfour_mar_apr = process_backup_data(backup1)
twentyfour_mar = process_backup_data(backup2)
twentyfour_mar_mid = process_backup_data(backup3)
twentythree_sep_mar = process_backup_data(backup4)

all_backup_data = pd.concat([
    twentyfour_apr_may,
    twentyfour_mar_apr,
    twentyfour_mar,
    twentyfour_mar_mid,
    twentythree_sep_mar
], ignore_index=True)



In [4]:
all_backup_data['Date'] = pd.to_datetime(all_backup_data['Date'], errors='coerce')

# Drop rows with NaT values in the Date column
all_backup_data = all_backup_data.dropna(subset=['Date'])

# Extract date part
all_backup_data['Date_only'] = all_backup_data['Date'].dt.date

# Split DataFrame by date and save each to a CSV file
for date in all_backup_data['Date_only'].unique():
    date_str = date.strftime('%Y-%m-%d')
    df_date = all_backup_data[all_backup_data['Date_only'] == date].copy()
    # Drop the Date_only column before saving
    df_date.drop(columns=['Date_only'], inplace=True)
    
    # Save to CSV
    df_date.to_csv(f'corrected_data_{date_str}.csv', index=False)

KeyboardInterrupt: 

## Part2 -- Recent Data Loading
### Reformatting Instruction
After 2024-07-03
- drag the original csv file into the folder
- change the start_date and end_date below
- run all the cells below
- Done!

In [4]:
# load recent SR data
start_date = datetime(2024, 7, 19)
end_date = datetime(2024, 7, 21)

In [5]:
# col names
column_names = ['Date+T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
col_after = ['Date', 'T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
col1 = ['T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
col2 = ['T1', 'T2', 'T3', 'T4', 'T5', 'T6', 'T7', 'T8', 'T9', 'T10', 'T11', 'T12', 'T13', 'T14', 'T15', 'F1', 'F2', 'F3', 'SR', 'A', 'Circ pump counter']
dataframes = {}
current_date = start_date

while current_date <= end_date:
    file_name = f'data_{current_date.strftime("%Y-%m-%d")}.csv'
    date_key = current_date.strftime("%Y-%m-%d")

    try:
        # load the data without using the first row as headers and specify the column names
        data = pd.read_csv(file_name, header=None, names=column_names)

        # check if 'T1' is concatenated with 'Date' and handle accordingly
        if '-' in data.iloc[0, 0]:
            # split 'Date+T1' into two separate columns 'Date' and 'T1'
            new_cols = data['Date+T1'].str.split(' T1:', expand=True)
            data['Date'] = new_cols[0]
            data['T1'] = new_cols[1].astype(float)
            data = data.drop(columns='Date+T1')
            # make sure 'T1' is correctly placed without affecting other columns
            data = data[col_after]
            for col in data[col1]:
                data[col] = data[col].str.split(':').str[1].astype(float)
        else:
            # separate 'Date' and 'T1' if not concatenated
            data= data.reset_index().rename(columns={'index': 'Date'})
            data = data.rename(columns={'Date+T1': 'T1'})
            data = data[col_after]
            for col in data[col2]:
                data[col] = data[col].str.split(':').str[1].astype(float)


        data.drop(data.index[:1], inplace=True)
        # store the DataFrame in the dictionary with the date as the key
        dataframes[date_key] = data
        
    except FileNotFoundError:
        print(f"File not found: {file_name}")
    except Exception as e:
        print(f"Error reading {file_name}: {e}")

    current_date += timedelta(days=1)

In [6]:
for date, df in dataframes.items():
    df.to_csv(f'corrected_data_{date}.csv', index=False)


## Temperature Column

In [ ]:
yst_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/yesterday?unitGroup=metric&key=FLBEV2T2WUAPQGP2MP25GFASV&contentType=json"
last_7_day_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/last7days?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
last_15_day_url =  "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/last15days?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
tmr_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/tomorrow?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
next_7_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/next7days?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
next_30_day_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/next30days?unitGroup=metric&key=FLBEV2T2WUAPQGP2MP25GFASV&contentType=json"
next_24_hours_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/next24hours?unitGroup=metric&key=HJ2JWKPKSPZKWD64FVT9F7A6Z&contentType=json"
today_url = "https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/today?include=fcst%2Cobs%2Chistfcst%2Cstats%2Cdays%2Chours&key=FLBEV2T2WUAPQGP2MP25GFASV&contentType=json"

url_start_date = "2024-07-05"  
url_end_date = "2024-07-07"    

self_define_url = f"https://weather.visualcrossing.com/VisualCrossingWebServices/rest/services/timeline/vancouver/{url_start_date}/{url_end_date}?unitGroup=metric&key=FLBEV2T2WUAPQGP2MP25GFASV&contentType=json"


In [ ]:
# tmr weather function
def get_tmr_weather_df():
    # URL setup for tomorrow's weather data
    tmr_date = datetime.now() + timedelta(days=1)
    tmr_response = requests.get(tmr_url)

    # Check if the request was successful
    if tmr_response.status_code == 200:
        # Parse the JSON data
        weather_data = tmr_response.json()
        print("Data retrieved successfully!")
        
        # File path setup for saving the data
        tmr_file_path = f"{tmr_date.strftime('%Y-%m-%d')}_weather_data.json"
        with open(tmr_file_path, 'w') as f:
            json.dump(weather_data, f, indent=4)
        print("Weather data saved successfully!")
        
        # Load and parse the JSON data into a DataFrame
        with open(tmr_file_path, 'r') as file:
            tmr_json_data = json.load(file)
        
        # Initialize an empty list to hold all hourly data
        tmr_hourly_data = []
        
        # Loop through each day in the data
        for day in tmr_json_data['days']:
            # Extract each 'hour' from 'hours' array and enrich it with 'day' level data
            for hour in day['hours']:
                hour.update({
                    'date': day['datetime'],
                    'tempmax': day['tempmax'],
                    'tempmin': day['tempmin'],
                    'feelslikemax': day['feelslikemax'],
                    'feelslikemin': day['feelslikemin'],
                    'sunrise': day['sunrise'],
                    'sunset': day['sunset'],
                    'moonphase': day['moonphase'],
                    'conditions': day['conditions'],
                    'description': day['description'],
                    'icon': day['icon']
                })
                tmr_hourly_data.append(hour)
        
        # Create a DataFrame from the hourly data
        tmr_hourly_df = pd.DataFrame(tmr_hourly_data)
        
        # Define the columns of interest
        columns_of_interest = [
            'date', 'datetime', 'temp', 'tempmax', 'tempmin', 'feelslike', 'feelslikemax', 'feelslikemin', 
            'dew', 'humidity', 'precip', 'precipprob', 'precipcover', 'preciptype', 'snow', 'snowdepth',
            'windgust', 'windspeed', 'winddir', 'pressure', 'cloudcover', 'visibility', 
            'solarradiation', 'solarenergy', 'uvindex', 'severerisk', 'sunrise', 'sunset', 
            'moonphase', 'conditions', 'description', 'icon'
        ]

        
        # Reindex and rename the DataFrame
        tmr_hourly_df = tmr_hourly_df.reindex(columns=columns_of_interest).rename(columns={"date": "Dates", "datetime": "Time"})
        latitude = 49.2827

        def get_declination(day_of_year):
            return 23.45 * np.sin(np.deg2rad(360 * (284 + day_of_year) / 365))

        def get_hour_angle(time, longitude):
            total_hours = time.hour + time.minute / 60 + time.second / 3600
            solar_noon = 12 - (longitude / 15)
            return 15 * (total_hours - solar_noon)

        def solar_elevation(latitude, declination, hour_angle):
            latitude_rad = np.deg2rad(latitude)
            declination_rad = np.deg2rad(declination)
            hour_angle_rad = np.deg2rad(hour_angle)
            elevation_rad = np.arcsin(np.sin(latitude_rad) * np.sin(declination_rad) +
                                    np.cos(latitude_rad) * np.cos(declination_rad) * np.cos(hour_angle_rad))
            return np.rad2deg(elevation_rad)

        tmr_hourly_df['Time'] = pd.to_datetime(tmr_hourly_df['Time'])
        tmr_hourly_df['Sun Angle'] = tmr_hourly_df['Time'].apply(
            lambda x: solar_elevation(
                latitude,
                get_declination(x.timetuple().tm_yday),
                get_hour_angle(x, longitude=-123.1207)
            )
        )
        return tmr_hourly_df
    else:
        print("Failed to retrieve data:", tmr_response.status_code)
        return None
    
tmr_weather_df = get_tmr_weather_df()
tmr_weather_df

# function to store weather prediction
def store_and_update_weather_prediction(tmr_weather_df):

    # Current date for the existing file and next day for the new file name
    current_date = datetime.now().strftime('%Y-%m-%d')
    next_date = (datetime.now() + timedelta(days=1)).strftime('%Y-%m-%d')
    
    # File paths
    existing_file_name = f'weather_prediction_till_{current_date}.csv'
    new_file_name = f'weather_prediction_till_{next_date}.csv'
    
    # Check if the current file exists
    if os.path.exists(existing_file_name):
        # Read the existing CSV file
        existing_data = pd.read_csv(existing_file_name)
        # Concatenate the new data
        updated_data = pd.concat([existing_data, tmr_weather_df], ignore_index=True)
        print("Existing data found and updated.")
    else:
        # If the file does not exist, just use the new data
        updated_data = tmr_weather_df
        print(f"No existing file found for today ({current_date}). A new file will be created.")
    
    # Save the updated data to the new file name
    updated_data.to_csv(new_file_name, index=False)
    
    # Delete the old file if it exists
    if os.path.exists(existing_file_name):
        os.remove(existing_file_name)
        print(f"Old file '{existing_file_name}' deleted successfully.")
    
    # Confirm the new file creation
    if os.path.exists(new_file_name):
        print(f"New file '{new_file_name}' created successfully with updated predictions.")
    else:
        print(f"Failed to create the new file '{new_file_name}'.")
        
store_and_update_weather_prediction(tmr_weather_df)

In [7]:
weather = pd.read_csv('weather_20240420_to_20240721.csv').drop(columns=['Unnamed: 0'])
weather = weather[['Dates', 'Time', 'temp', 'solarradiation']]
weather

,Dates,Time,temp,solarradiation
0,2024-04-20,00:00:00,11.0,0.0
1,2024-04-20,01:00:00,9.8,0.0
2,2024-04-20,02:00:00,9.8,0.0
3,2024-04-20,03:00:00,9.4,0.0
4,2024-04-20,04:00:00,9.0,0.0
...,...,...,...,...
2251,2024-07-21,19:00:00,20.4,69.0
2252,2024-07-21,20:00:00,20.2,37.0
2253,2024-07-21,21:00:00,18.9,0.0
2254,2024-07-21,22:00:00,19.1,0.0


In [8]:
import os

folder_path = '.'
files = os.listdir(folder_path)

total_weather = weather
total_weather['DateTime'] = pd.to_datetime(total_weather['Dates'].astype(str) + ' ' + total_weather['Time'].astype(str))
total_weather.drop(['Dates', 'Time'], axis=1, inplace=True)


for file in files:
    if file.startswith("corrected_data_"):
        file_path = os.path.join(folder_path, file)
        corrected_data = pd.read_csv(file_path)
        
        # Parse the 'Date' column to datetime, trying to infer the format automatically
        corrected_data['Date'] = pd.to_datetime(corrected_data['Date'], infer_datetime_format=True)
        
        # Round down to the nearest hour for merging
        corrected_data['Date_hour'] = corrected_data['Date'].dt.floor('H')
        
        # Merge with the weather data on the rounded hour
        merged_data = pd.merge(corrected_data, total_weather, how='left', left_on='Date_hour', right_on='DateTime')

        # Conditionally replace SR values that are under 80 with the solarradiation value
        mask = merged_data['SR'] <= 80
        merged_data.loc[mask, 'SR'] = merged_data.loc[mask, 'solarradiation']
        
        # Drop unnecessary columns
        merged_data.drop(['Date_hour', 'solarradiation', 'DateTime'], axis=1, inplace=True)

        # Save the merged and modified DataFrame to a new CSV file
        new_file_name = 'weather_added_' + file
        new_file_path = os.path.join(folder_path, new_file_name)
        merged_data.to_csv(new_file_path, index=False)

print("All files processed and saved with added weather data.")

/var/folders/h0/8gdh0lxd4ynbp3ypwp6nwxmc0000gn/T/ipykernel_4788/453782872.py:17: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  corrected_data['Date'] = pd.to_datetime(corrected_data['Date'], infer_datetime_format=True)
/var/folders/h0/8gdh0lxd4ynbp3ypwp6nwxmc0000gn/T/ipykernel_4788/453782872.py:20: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  corrected_data['Date_hour'] = corrected_data['Date'].dt.floor('H')
/var/folders/h0/8gdh0lxd4ynbp3ypwp6nwxmc0000gn/T/ipykernel_4788/453782872.py:17: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can

All files processed and saved with added weather data.
